# Exploración de Qbeast sobre Delta Lake

Este notebook es independiente del benchmark de `Iceberg/dremio-file-ingestion-practice`: Qbeast no se integra con catálogos Iceberg/Nessie, requiere Delta Lake y su propio `QbeastCatalog` como `spark_catalog`. Aquí escribimos la misma tabla como Delta "plano" y como Qbeast (indexado), y comparamos cuántos ficheros lee cada uno ante una consulta filtrada.

**Dataset:** NYC Yellow Taxi Trip Records (Enero 2024).

## 1. Sesión de Spark

La configuración (paquetes de Delta y Qbeast, extensiones, catálogo) se carga desde `spark-defaults.conf`, copiado en la imagen.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date
import time

spark = (
    SparkSession.builder
    .appName("Qbeast-Delta-Exploracion")
    .getOrCreate()
)

print(f"Spark session creada. Versión: {spark.version}")

## 2. Descarga y preparación del dataset

Spark no puede leer un `https://` directamente (falla con `UnsupportedOperationException: hasn't implemented listStatus`), así que descargamos el fichero a disco local del contenedor primero.

In [ ]:
import os
import requests

DATASET_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
LOCAL_PATH = "/home/jovyan/work/data/yellow_tripdata_2024-01.parquet"

os.makedirs(os.path.dirname(LOCAL_PATH), exist_ok=True)

if not os.path.exists(LOCAL_PATH):
    print(f"Descargando {DATASET_URL} ...")
    resp = requests.get(DATASET_URL, timeout=120)
    resp.raise_for_status()
    with open(LOCAL_PATH, "wb") as f:
        f.write(resp.content)
    print(f"Descargado en {LOCAL_PATH} ({os.path.getsize(LOCAL_PATH):,} bytes)")
else:
    print(f"Ya existe {LOCAL_PATH}, se omite descarga.")

In [ ]:
raw_df = spark.read.parquet(f"file://{LOCAL_PATH}")

df = (
    raw_df.select(
        col("tpep_pickup_datetime"),
        col("tpep_dropoff_datetime"),
        col("passenger_count").cast("int"),
        col("trip_distance").cast("double"),
        col("PULocationID").cast("int"),
        col("DOLocationID").cast("int"),
        col("fare_amount").cast("double"),
        col("tip_amount").cast("double"),
        col("total_amount").cast("double"),
    )
    .withColumn("pickup_date", to_date(col("tpep_pickup_datetime")))
    .filter(col("fare_amount") > 0)
    .dropna()
)

df.cache()
print(f"Registros preparados: {df.count():,}")
df.printSchema()

## 3. Escritura: Delta plano (referencia) vs. Qbeast (indexado)

In [ ]:
DELTA_PATH = "/home/jovyan/warehouse/delta/nyc_taxi_delta"
QBEAST_PATH = "/home/jovyan/warehouse/qbeast/nyc_taxi_qbeast"

start = time.time()
(df.write
 .format("delta")
 .mode("overwrite")
 .save(DELTA_PATH))
print(f"Delta plano escrito en {time.time() - start:.2f}s")

In [ ]:
columns_to_index = "PULocationID,DOLocationID,fare_amount"

start = time.time()
(df.write
 .format("qbeast")
 .mode("overwrite")
 .option("columnsToIndex", columns_to_index)
 .option("cubeSize", 50000)
 .save(QBEAST_PATH))
print(f"Qbeast escrito en {time.time() - start:.2f}s (columnas indexadas: {columns_to_index})")

## 4. Comparativa: ficheros leídos ante una consulta filtrada

El valor real de Qbeast es reducir los datos leídos (*data skipping*) en consultas selectivas sobre las columnas indexadas, no necesariamente la velocidad de escritura.

In [ ]:
FILTER_CONDITION = "PULocationID = 161 AND fare_amount > 20"

delta_filtered = spark.read.format("delta").load(DELTA_PATH).where(FILTER_CONDITION)
qbeast_filtered = spark.read.format("qbeast").load(QBEAST_PATH).where(FILTER_CONDITION)

delta_count = delta_filtered.count()
delta_files = len(delta_filtered.inputFiles())

qbeast_count = qbeast_filtered.count()
qbeast_files = len(qbeast_filtered.inputFiles())

print(f"Filtro: {FILTER_CONDITION}\n")
print(f"Delta plano -> filas: {delta_count:,} | ficheros leídos: {delta_files}")
print(f"Qbeast      -> filas: {qbeast_count:,} | ficheros leídos: {qbeast_files}")

In [ ]:
print("--- Plan físico Delta plano ---")
delta_filtered.explain("formatted")

print("\n--- Plan físico Qbeast ---")
qbeast_filtered.explain("formatted")

## 5. Inspección del índice de Qbeast

La API `QbeastTable` es Scala; se accede vía el puente `py4j` de PySpark (`spark._jvm`).

In [ ]:
qbeast_table = spark._jvm.io.qbeast.spark.QbeastTable.forPath(spark._jsparkSession, QBEAST_PATH)

print("Columnas indexadas:", qbeast_table.indexedColumns())
print("Revisión más reciente:", qbeast_table.latestRevisionID())
print("\nMétricas del índice:")
print(qbeast_table.getIndexMetrics())

## 6. Limpieza

In [ ]:
df.unpersist()
spark.stop()
print("Sesión de Spark detenida.")